# NVIDIA 原始 SEC Data Center 实际收入：来源覆盖预检

只读已封存的 63 份原始盈利公告；核对来源身份和文本覆盖，不抽取数值或读取 SOXX 收益。


In [1]:
import sys,json,hashlib,re,collections,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from bs4 import BeautifulSoup
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_nvda_datacenter_original_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];parent=root/src['parent_card'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_HTTP_GETs','market_outcome_reads','new_model_fits','native_accounts','Qlib_Recorders'))
repair=json.loads((card/'repair_1_plan.json').read_text());assert repair['frozen'] and repair['parent_plan_sha256']==sha(card/'plan.json') and repair['source_quarter_events']==63
assert sha(parent/'quarter_event_adjudication_result.json')==src['selected_quarter_event_adjudication_sha256'] and sha(parent/'item202_inventory_result.json')==src['Item202_inventory_sha256']
inventory=json.loads((parent/'item202_inventory_result.json').read_text());assert inventory['NAS_archive_sha256']==src['all_original_raw_NAS_archive_sha256'] and inventory['NAS_remote_and_scoped_restore_verified']
events=json.loads((parent/'quarter_event_adjudication_result.json').read_text())['selected_events'];nvda=[x for x in events if x['issuer']=='NVDA'];assert len(nvda)==src['selected_NVDA_events']==63
rows=[]
for x in nvda:
 p=root/src['source_root']/src['raw_file_template'].format(accession=x['accession']);assert p.is_file() and sha(p)==x['raw_sha256']
 full=p.read_text(errors='replace')
 docs=[m.group(1) for m in re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>',full,re.I|re.S)]
 expected='EX-95.1' if x['accession']=='0001045810-19-000168' else 'EX-99.1'
 hits=[doc for doc in docs if re.search(r'<TYPE>\s*'+re.escape(expected)+r'(?:\s|<)',doc,re.I)]
 assert len(hits)==1,(x['accession'],expected,len(hits))
 if expected=='EX-95.1':assert re.search(r'<DESCRIPTION>Q3FY20 PRESS RELEASE',hits[0],re.I)
 body=re.search(r'<TEXT>(.*?)</TEXT>',hits[0],re.I|re.S);assert body
 text=BeautifulSoup(body.group(1),'html.parser').get_text(' ',strip=True);text=re.sub(r'\s+',' ',text)
 spans=[m.start() for m in re.finditer(r'data\s*center|datacenter',text,re.I)]
 numeric=[];yoy=[]
 for pos in spans:
  window=text[pos:pos+250]
  if re.search(r'(?i)revenue',window) and re.search(r'\$\s*[0-9][0-9,.]*\s*(?:million|billion)?',window):numeric.append(pos)
  if re.search(r'(?i)(?:year.earlier|year.over.year|year.ago|y/y|yoy)',window):yoy.append(pos)
 rows.append({'announcement_year':x['announcement_year'],'slot':x['slot'],'filing_date':x['filing_date'],'accession':x['accession'],'raw_sha256':x['raw_sha256'],'selected_original_document_type':expected,'Data_Center_text_hits':len(spans),'nearby_revenue_dollar_candidates':len(numeric),'nearby_YoY_word_candidates':len(yoy),'numeric_semantics_qualified':False,'first_publication_qualified':False})
assert len(rows)==63 and len({x['accession'] for x in rows})==63
years={str(y):{'events':len(g),'term_present':sum(r['Data_Center_text_hits']>0 for r in g),'revenue_dollar_candidate':sum(r['nearby_revenue_dollar_candidates']>0 for r in g),'YoY_word_candidate':sum(r['nearby_YoY_word_candidates']>0 for r in g)} for y in sorted({x['announcement_year'] for x in rows}) for g in [[r for r in rows if r['announcement_year']==y]]}
result={'decision':'original_SEC_NVDA_Data_Center_text_coverage_only_numeric_semantics_unqualified','plan_sha256':sha(card/'plan.json'),'selected_NVDA_events':63,'raw_hash_verified_events':63,'term_present_events':sum(r['Data_Center_text_hits']>0 for r in rows),'revenue_dollar_candidate_events':sum(r['nearby_revenue_dollar_candidates']>0 for r in rows),'YoY_word_candidate_events':sum(r['nearby_YoY_word_candidates']>0 for r in rows),'first_revenue_dollar_candidate':next((r['filing_date'] for r in rows if r['nearby_revenue_dollar_candidates']>0),None),'year_counts':years,'events':rows,'numeric_factor_values_extracted':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_publication_qualified':False}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
settings=_settings(root);files=[card/'plan.json',card/'result.json']
with tempfile.TemporaryDirectory(prefix='nvda-dc-source-') as tmp:
 a=Path(tmp)/'source.tar.gz'
 with tarfile.open(a,'w:gz') as out:
  for p in files:out.add(p,arcname=str(p.relative_to(root)))
 h=sha(a);response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+h,upload=a,executor_id='soxl-minute-research');rec=response['artifact'];assert rec['sha256']==h and rec['size_bytes']==a.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+h[:2]+'/'+h[2:4]+'/'+h;restore=Path(tmp)/'restore.tar.gz'
 with restore.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and sha(restore)==h
 with tarfile.open(restore,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(p.relative_to(root)) for p in files}
  for p in files:assert hashlib.sha256(z.extractfile(names[str(p.relative_to(root))]).read()).hexdigest()==sha(p)
receipt={'status':'source_audit_NAS_scoped_restore_verified','sha256':h,'size_bytes':rec['size_bytes'],'nas_path':remote,'members':{str(p.relative_to(root)):sha(p) for p in files},'verified':True};(card/'nas_source_archive.json').write_text(json.dumps(receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','raw_parent_archive_sha256':src['all_original_raw_NAS_archive_sha256'],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Text-window candidate is not a validated segment revenue value or year-over-year comparison.','SEC filing date is not proved to be the first issuer release time.','No Qlib factor, future SOXX outcome or account was produced.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'term_present':result['term_present_events'],'revenue_candidate':result['revenue_dollar_candidate_events'],'YoY_candidate':result['YoY_word_candidate_events'],'first_candidate':result['first_revenue_dollar_candidate'],'years':years,'nas_sha256':h},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "term_present": 50,
  "revenue_candidate": 31,
  "YoY_candidate": 30,
  "first_candidate": "2014-05-08",
  "years": {
    "2011": {
      "events": 4,
      "term_present": 0,
      "revenue_dollar_candidate": 0,
      "YoY_word_candidate": 0
    },
    "2012": {
      "events": 4,
      "term_present": 0,
      "revenue_dollar_candidate": 0,
      "YoY_word_candidate": 0
    },
    "2013": {
      "events": 4,
      "term_present": 2,
      "revenue_dollar_candidate": 0,
      "YoY_word_candidate": 0
    },
    "2014": {
      "events": 4,
      "term_present": 3,
      "revenue_dollar_candidate": 1,
      "YoY_word_candidate": 1
    },
    "2015": {
      "events": 4,
      "term_present": 2,
      "revenue_dollar_candidate": 0,
      "YoY_word_candidate": 0
    },
    "2016": {
      "events": 4,
      "term_present": 4,
      "revenue_dollar_candidate": 1,
      "YoY_word_candidate": 0
    },
    "2017": {
      "events": 4,
      "term_present": 4,
      "revenue_dollar_candid

# NVIDIA 原始 SEC Data Center 实际收入：来源覆盖预检

只读已封存的 **63 份** NVIDIA 季度盈利公告 SEC 原件，逐件核对 accession 和原字节 SHA-256；父来源及本卡小审计均在 Yanxuan NAS 归档、限定恢复。2011、2012 年的八份公告没有 Data Center/datacenter 字样；全部 63 份中 **50 份**出现该术语，简单的“术语附近收入金额”文本探针命中 **31 份**。这仅是定位线索，不是 31 个已认证的收入数值或因子。

来源结构解释了探针低估：2016–2019 年的实际收入常放在同一原始申报的 CFO commentary 附件 `EX-99.2` 的 “Revenue by Market Platform” 表，而新闻稿 `EX-99.1` 多为业务叙述；2019 年一份新闻稿原件标为 `EX-95.1`。2020 年后的公告逐步在新闻稿或表中直接给出 Data Center 收入及同比。若研究“AI 数据中心需求”，应固定**原始表中的当前季与上年同期金额**，保留早期未知和名称从 `Datacenter` 到 `Data Center` 的变化，不能把整段“提到 Data Center”当成增长值。

首次执行把整份 SEC submission（含图像/ZIP）送入 HTML 解析器，在第一份原件遇到无效字符引用而中止；没有持久化结果。一次机械修复仅选择新闻稿 `DOCUMENT/TEXT`，62 份 `EX-99.1` 加一份有原始新闻稿说明的 `EX-95.1`，原 accession 与原字节不变。失败尝试读了 1 份，修复执行又读 63 份，**累计 64 次原件读取，超过冻结的 63 次预算 1 次**；错误收据、修复计划和预算收据均保留，不再重跑。

本卡 **0 数值因子、0 SOXX 收益读取、0 拟合、0 账户**。决定：`original_SEC_NVDA_Data_Center_text_coverage_only_numeric_semantics_unqualified`。下一步另行冻结 CFO 原始表的逐季数值/单位/同比及 SEC 接收时钟审计；即使来源过门，还要先检验其相对原 Stage1 39 项的增量，再进入完整 0bp 账户。
